# RAG Assignment: Baseline Bi-Encoder Retrieval vs. Cross-Encoder Re-Ranking
**Document:** `ICT_policy_gov_Telangana.pdf` (G.O.Ms.No.09, ICT Policy 2021-2026, Govt. of Telangana)

| Component | Choice |
|---|---|
| Bi-Encoder (dense retriever) | `sentence-transformers/all-MiniLM-L6-v2`, FAISS `IndexFlatIP` over L2-normalised vectors (inner product = cosine) |
| Cross-Encoder | `cross-encoder/ms-marco-MiniLM-L-6-v2` |
| LLM | `Qwen/Qwen2.5-7B-Instruct` (open weights, greedy decoding). Auto-configured: fp16/bf16 if the GPU has >= 20 GB (L4/A100), 4-bit NF4 on smaller GPUs (free Colab T4); falls back to the 3B model on CPU. |

**GPU optimisations:** fp16 encoders, TF32 matmuls, SDPA attention, `inference_mode`, large batches for corpus embedding, a hand-written batched fp16 Cross-Encoder scorer (raw logits, no fp16-saturating sigmoid), CUDA-synchronised timers and GPU warm-up.

Pipeline 1: query -> Top-10 (bi-encoder) -> Top-3 -> LLM -> **Answer 1**.
Pipeline 2: same Top-10 -> Cross-Encoder re-sort -> Top-3 -> LLM (identical prompt) -> **Answer 2**.

## 0. Colab setup
Runtime -> Change runtime type -> **GPU** (a T4 is enough). Upload `ICT_policy_gov_Telangana.pdf` to the Colab file panel. The scanned pages are OCR'd with **RapidOCR (PP-OCRv6, medium det + rec models)**; the models are downloaded automatically on first use.

In [ ]:
!pip -q install sentence-transformers faiss-cpu pymupdf accelerate bitsandbytes pandas "rapidocr>=3.10.0" onnxruntime

In [ ]:
import json, os, re, time, textwrap, warnings, shutil
import numpy as np, pandas as pd, torch, faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120, "display.width", 200)

PDF_PATH   = "ICT_policy_gov_Telangana.pdf"
OCR_CACHE  = "ict_policy_ocr_rapidocr.json"   # cache of the RapidOCR output
FORCE_OCR  = False                              # True = ignore the cache and OCR again
BI_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CE_NAME    = "cross-encoder/ms-marco-MiniLM-L-6-v2"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
TOP_K, TOP_N = 10, 3

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True      # TF32 on Ampere+ (no effect on T4)
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    LLM_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16   # bf16 only on Ampere+ (T4 emulates it slowly)
    ENC_DTYPE = torch.float16                          # small encoders: fp16 is fast and accurate enough
    LLM_NAME  = "Qwen/Qwen2.5-7B-Instruct"
    USE_4BIT  = GPU_MEM_GB < 20                        # 7B fp16 needs ~16 GB; T4 (15 GB) -> NF4 4-bit
else:
    print("WARNING: no GPU found. Runtime -> Change runtime type -> GPU. Falling back to CPU + 3B model (slow).")
    GPU_NAME, GPU_MEM_GB = None, 0
    LLM_DTYPE = ENC_DTYPE = torch.float32
    LLM_NAME, USE_4BIT = "Qwen/Qwen2.5-3B-Instruct", False
print(f"device={DEVICE} gpu={GPU_NAME} mem={GPU_MEM_GB:.1f}GB llm={LLM_NAME} 4bit={USE_4BIT} llm_dtype={LLM_DTYPE}")

## 1. Text extraction
The PDF is a **scanned document (22 page images, no text layer)** – `pypdf`/`PyMuPDF` return 0 characters. So the text has to be OCR'd first.
Pages are rendered at 200 dpi and OCR'd with **RapidOCR** using the **PP-OCRv6 `medium`** text-detection and text-recognition models (ONNX Runtime). The `medium` size exists only for PP-OCRv6, which needs `rapidocr >= 3.10`. Detected text boxes are grouped into lines (top-to-bottom, left-to-right) to rebuild reading order. The text is cached in `ict_policy_ocr_rapidocr.json` so a re-run skips OCR (set `FORCE_OCR = True` to redo it), and it is also exported with the results. If a gold-chunk regex in Section 5 matches nothing, adjust it to the OCR text (the cell warns you).

In [ ]:
from rapidocr import RapidOCR, ModelType, OCRVersion, LangDet, LangRec

def build_ocr():
    return RapidOCR(params={
        "Det.ocr_version": OCRVersion.PPOCRV6, "Det.model_type": ModelType.MEDIUM, "Det.lang_type": LangDet.EN,
        "Rec.ocr_version": OCRVersion.PPOCRV6, "Rec.model_type": ModelType.MEDIUM, "Rec.lang_type": LangRec.EN,
        "Global.max_side_len": 4000,        # default 2000 would down-scale a 200-dpi page and blur small print
        "Global.log_level": "warning"})

def boxes_to_text(boxes, txts, line_tol=0.5):
    # group detected boxes into text lines (same vertical centre), then read each line left-to-right
    if boxes is None or len(txts) == 0: return ""
    items = []
    for b, t in zip(np.asarray(boxes), txts):
        ys, xs = b[:, 1], b[:, 0]
        items.append(dict(x=float(xs.min()), yc=float(ys.mean()), h=float(ys.max() - ys.min()), t=t))
    items.sort(key=lambda d: d["yc"])
    lines, cur = [], [items[0]]
    for it in items[1:]:
        ref_h = np.median([c["h"] for c in cur])
        if abs(it["yc"] - np.mean([c["yc"] for c in cur])) <= line_tol * ref_h: cur.append(it)
        else: lines.append(cur); cur = [it]
    lines.append(cur)
    return "\n".join(" ".join(c["t"] for c in sorted(l, key=lambda d: d["x"])) for l in lines)

def ocr_pdf(pdf_path, dpi=200):
    import pymupdf
    engine, pages = build_ocr(), []
    for n, p in enumerate(pymupdf.open(pdf_path), 1):
        pix = p.get_pixmap(dpi=dpi, alpha=False)
        img = np.frombuffer(pix.samples, np.uint8).reshape(pix.h, pix.w, 3)[:, :, ::-1].copy()   # RGB -> BGR
        res = engine(img)
        pages.append(boxes_to_text(res.boxes, res.txts or ()))
        print(f"OCR page {n}: {len(pages[-1])} chars, {res.elapse:.1f}s")
    return pages

if os.path.exists(OCR_CACHE) and not FORCE_OCR:
    pages = json.load(open(OCR_CACHE))
else:
    t0 = time.perf_counter(); pages = ocr_pdf(PDF_PATH); json.dump(pages, open(OCR_CACHE, "w"), indent=1)
    print(f"OCR finished in {time.perf_counter()-t0:.0f}s")
print(len(pages), "pages,", sum(map(len, pages)), "characters")
print(pages[4][:600])

## 2. Chunking (shared corpus for both pipelines)
Sentence-aware packing: sentences are accumulated up to ~450 characters, with a 1-sentence overlap between neighbouring chunks. Page headers/footers ("Page x of 22") are removed. Small chunks keep each rule/condition localised, so a single rule can be split over several chunks – which is exactly the situation where re-ranking matters.

In [ ]:
def make_chunks(pages, max_chars=450, overlap_sents=1):
    sents = []
    for pno, t in enumerate(pages, 1):
        t = re.sub(r"Page \d+ of \d+", "", t)
        t = re.sub(r"\s*\n\s*", " ", t)
        for s in re.split(r"(?<=[.:;?!])\s+(?=[A-Z0-9•(])", t):
            if s.strip(): sents.append((pno, s.strip()))
    chunks, cur = [], []
    for pno, s in sents:
        if cur and sum(len(x[1]) + 1 for x in cur) + len(s) > max_chars:
            chunks.append({"page": cur[0][0], "text": " ".join(x[1] for x in cur)})
            cur = cur[-overlap_sents:]
        cur.append((pno, s))
    if cur: chunks.append({"page": cur[0][0], "text": " ".join(x[1] for x in cur)})
    for i, c in enumerate(chunks): c["id"] = i
    return chunks

chunks = make_chunks(pages)
texts = [c["text"] for c in chunks]
print(len(chunks), "chunks; mean length", int(np.mean([len(t) for t in texts])), "chars")
print(chunks[63])   # chunk ids may shift slightly with a different OCR engine

## 3. Models and vector index (Bi-Encoder + FAISS)

In [ ]:
def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

# --- Bi-Encoder (fp16 on GPU) ---
bi = SentenceTransformer(BI_NAME, device=DEVICE)
if DEVICE == "cuda": bi.half()

# --- Cross-Encoder: plain HF model, batched fp16 scoring, raw logits ---
class CrossEncoderScorer:
    def __init__(self, name, device, dtype, max_length=512):
        self.tok = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForSequenceClassification.from_pretrained(name, torch_dtype=dtype).to(device).eval()
        self.device, self.max_length = device, max_length
    @torch.inference_mode()
    def score(self, query, passages, batch_size=128):
        out = []
        for i in range(0, len(passages), batch_size):
            enc = self.tok([query] * len(passages[i:i + batch_size]), passages[i:i + batch_size], padding=True,
                           truncation="only_second", max_length=self.max_length, return_tensors="pt").to(self.device)
            out.append(self.model(**enc).logits[:, 0].float().cpu())   # raw logits; sigmoid would saturate in fp16
        return torch.cat(out).numpy()
ce = CrossEncoderScorer(CE_NAME, DEVICE, ENC_DTYPE)

# --- index the corpus: one large batched GPU pass, then a FAISS inner-product index ---
sync(); t0 = time.perf_counter()
emb = bi.encode(texts, batch_size=256, convert_to_numpy=True, show_progress_bar=False).astype("float32")
faiss.normalize_L2(emb)                       # unit vectors -> inner product == cosine similarity
index = faiss.IndexFlatIP(emb.shape[1])       # exact search; with ~300 vectors a CPU flat index takes microseconds
index.add(emb)
sync(); print(f"indexed {index.ntotal} chunks, dim={emb.shape[1]} in {time.perf_counter()-t0:.2f}s")

In [ ]:
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm_kwargs = dict(attn_implementation="sdpa")
if DEVICE == "cuda": llm_kwargs["device_map"] = {"": 0}
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    llm_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                           bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
else:
    llm_kwargs["torch_dtype"] = LLM_DTYPE
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, **llm_kwargs).eval()
if DEVICE == "cpu": llm = llm.to(DEVICE)
print(f"LLM loaded, GPU memory in use: {torch.cuda.memory_allocated()/1e9:.1f} GB" if DEVICE == "cuda" else "LLM loaded on CPU")

SYSTEM = ("You are a careful assistant answering questions about the Telangana ICT Policy 2021-2026. "
          "Use ONLY the numbered context passages. If the passages do not contain the answer, say so. "
          "Answer in at most 3 sentences and include any conditions or exceptions mentioned in the passages.")

def build_prompt(query, ctx_chunks):
    ctx = "\n\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(ctx_chunks))
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {query}\nAnswer:"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def generate(query, ctx_chunks, max_new_tokens=160):
    inputs = tok(build_prompt(query, ctx_chunks), return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None,
                       pad_token_id=tok.eos_token_id, use_cache=True)
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## 4. The two retrieval pipelines (with timing)
* `baseline_retrieve` – embed the query, FAISS Top-10.
* `rerank` – Cross-Encoder scores all 10 `(query, chunk)` pairs jointly, re-sorts descending.
* Both pipelines then take **Top-3** and use the *same* `build_prompt` / generation settings.

In [ ]:
def baseline_retrieve(query, k=TOP_K):
    sync(); t0 = time.perf_counter()
    q = bi.encode([query], convert_to_numpy=True, show_progress_bar=False).astype("float32")
    faiss.normalize_L2(q)
    scores, ids = index.search(q, k)
    dt = time.perf_counter() - t0                  # encode() already copies to host, i.e. the GPU has finished
    return ids[0].tolist(), scores[0].tolist(), dt

def rerank(query, cand_ids):
    sync(); t0 = time.perf_counter()
    ce_scores = ce.score(query, [texts[i] for i in cand_ids], batch_size=len(cand_ids))   # all 10 pairs in one GPU batch
    order = np.argsort(-ce_scores, kind="stable")
    dt = time.perf_counter() - t0
    return [cand_ids[j] for j in order], [float(ce_scores[j]) for j in order], dt

# GPU warm-up (CUDA context, cuDNN autotune, kernel loading) so it is not counted in the latency table
for _ in range(5):
    i_, _, _ = baseline_retrieve("warm up"); rerank("warm up", i_)
_ = generate("warm up", [chunks[0], chunks[1], chunks[2]], max_new_tokens=16)
sync()

## 5. Stress-test queries
`gold` is a regex for the **evidence phrase** that must be in the passages for a correct answer; the gold chunk(s) are all chunks that contain it (found automatically below). Categories follow the assignment brief.

In [ ]:
QUERIES = [
 dict(id="Q1", cat="Keyword trap / distractor",
      q="Who bears the cost of skilling workers for the electronics industry?",
      gold=r"on a no-cost basis",
      truth="TASK provides the skilled workforce on a no-cost basis (govt. bears it)."),
 dict(id="Q2", cat="Synonym / vital chunk buried",
      q="What other ways can citizens lodge complaints besides the mobile app?",
      gold=r"Telephone, SMS, WhatsApp",
      truth="Centralised call centre: telephone, SMS, WhatsApp, email, post, paper."),
 dict(id="Q3", cat="Exceptions & corner cases (who the relaxation applies to)",
      q="Do tender relaxations apply to all bidders or only to certain enterprises?",
      gold=r"Relax qualification norms",
      truth="Relaxations (experience, turnover, EMD, performance guarantee) are for SMEs / MSMEs only."),
 dict(id="Q4", cat="Negation & constraints",
      q="Is Telangana not planning to relax the norms for SMEs?",
      gold=r"Relax qualification norms",
      truth="It is planning to relax them (experience, turnover, EMD, performance guarantee)."),
 dict(id="Q5", cat="Negation & constraints",
      q="Which citizen services will still need people to visit a government office?",
      gold=r"No G2C service shall mandate physical presence",
      truth="None - the target is that no G2C service shall mandate physical presence."),
 dict(id="Q6", cat="Other: vital chunk buried at rank 7",
      q="Which departments must set aside budget for emerging technology applications?",
      gold=r"special budget for IT and Emerging",
      truth="Each user department allocates a special budget for IT and emerging technologies."),
 dict(id="Q7", cat="Role / eligibility limits",
      q="Who is not eligible for IT park status?",
      gold=r"Third-party developers",
      truth="Eligible: 4 categories (govt-allotted company, developers via auction/private, individual landowners, third-party developers) with Annexure-1 documents."),
 dict(id="Q8", cat="Control (hard to break): exception in same chunk",
      q="Does the 60:40 ratio on government-allotted land have any exception?",
      gold=r"unless there are different terms",
      truth="Yes: unless different terms are specified in the allotment MOU."),
 dict(id="Q9", cat="Control: reranker can also hurt",
      q="Is blockchain used for any welfare scheme?",
      gold=r"Stree Nidhi is enhanced through the use of technologies like blockchain",
      truth="Stree Nidhi uses blockchain and AI to better support SHGs (also beneficiary verification)."),
]
for Q in QUERIES:
    Q["gold_ids"] = [c["id"] for c in chunks if re.search(Q["gold"], c["text"], re.I | re.S)]
    print(Q["id"], Q["gold_ids"], "|", Q["q"])
    if not Q["gold_ids"]: print("   WARNING: gold regex matched no chunk - adjust it to the OCR text")

In [ ]:
def run_query(Q, with_llm=True):
    base_ids, base_scores, t_base = baseline_retrieve(Q["q"])
    rr_ids, rr_scores, t_rr = rerank(Q["q"], base_ids)
    r = dict(Q, base_ids=base_ids, base_scores=base_scores, rr_ids=rr_ids, rr_scores=rr_scores, t_base=t_base, t_rr=t_rr)
    r["base_gold_rank"] = {g: (base_ids.index(g) + 1 if g in base_ids else None) for g in Q["gold_ids"]}
    r["rr_gold_rank"]   = {g: (rr_ids.index(g) + 1 if g in rr_ids else None) for g in Q["gold_ids"]}
    if with_llm:
        t0 = time.perf_counter(); r["ans1"] = generate(Q["q"], [chunks[i] for i in base_ids[:TOP_N]]); sync(); r["t_gen1"] = time.perf_counter() - t0
        t0 = time.perf_counter(); r["ans2"] = generate(Q["q"], [chunks[i] for i in rr_ids[:TOP_N]]);   sync(); r["t_gen2"] = time.perf_counter() - t0
    return r

results = [run_query(Q) for Q in QUERIES]

## 6. Rank shifts (Top-10 baseline -> Cross-Encoder)

In [ ]:
fmt = lambda d: ", ".join(f"#{g}: " + (f"rank {r}" if r else "not in top-10") for g, r in d.items())
rows = []
for r in results:
    best_b = min([x for x in r["base_gold_rank"].values() if x] or [None], default=None)
    best_r = min([x for x in r["rr_gold_rank"].values() if x] or [None], default=None)
    rows.append({"Q": r["id"], "category": r["cat"], "gold chunk(s): baseline rank": fmt(r["base_gold_rank"]),
                 "-> after re-rank": fmt(r["rr_gold_rank"]),
                 "best gold in Top-3 (base)": best_b is not None and best_b <= 3,
                 "best gold in Top-3 (rerank)": best_r is not None and best_r <= 3})
rank_df = pd.DataFrame(rows)
rank_df

In [ ]:
for r in results:
    print("="*110); print(r["id"], "|", r["cat"]); print("Q:", r["q"]); print("Expected:", r["truth"])
    print("Gold chunks:", r["gold_ids"])
    print("Baseline Top-10 ids:", r["base_ids"]); print("Re-ranked  ids     :", r["rr_ids"])
    print("\nBaseline Top-3 (cosine):")
    for i, s in zip(r["base_ids"][:3], r["base_scores"][:3]): print(f"  #{i:<3} p.{chunks[i]['page']:<2} cos={s:.3f} {'<-- GOLD' if i in r['gold_ids'] else ''}\n      {texts[i][:230]}")
    print("Re-ranked Top-3 (cross-encoder logit):")
    for i, s in zip(r["rr_ids"][:3], r["rr_scores"][:3]): print(f"  #{i:<3} p.{chunks[i]['page']:<2} ce={s:.2f} {'<-- GOLD' if i in r['gold_ids'] else ''}\n      {texts[i][:230]}")
    print("\nAnswer 1 (baseline):", textwrap.fill(r["ans1"], 105)); print("Answer 2 (re-rank) :", textwrap.fill(r["ans2"], 105))

## 7. Latency profiling

In [ ]:
# Retrieval-stage latency averaged over repeated calls (retrieval is fast, so single runs are noisy).
REPS = 20
lat = []
for Q in QUERIES:
    tb, tr = [], []
    for _ in range(REPS):
        ids_, _, a = baseline_retrieve(Q["q"]); _, _, b = rerank(Q["q"], ids_); tb.append(a); tr.append(b)
    lat.append((np.mean(tb) * 1000, np.mean(tr) * 1000))
lat_df = pd.DataFrame({
    "Q": [r["id"] for r in results],
    "Baseline retrieval (ms)": [round(a, 1) for a, _ in lat],
    "Cross-Encoder re-rank of 10 (ms)": [round(b, 1) for _, b in lat],
    "Pipeline 2 retrieval total (ms)": [round(a + b, 1) for a, b in lat],
    "LLM gen, answer 1 (s)": [round(r["t_gen1"], 1) for r in results],
    "LLM gen, answer 2 (s)": [round(r["t_gen2"], 1) for r in results]})
lat_df.loc["mean"] = ["mean"] + lat_df.iloc[:, 1:].mean().round(1).tolist()
lat_df

In [ ]:
# Why not Cross-Encode everything? Measure the cost of scoring ALL chunks (and extrapolate).
q = QUERIES[0]["q"]
def time_ce(n, reps=5):
    ts = []
    for _ in range(reps):
        sync(); t0 = time.perf_counter(); ce.score(q, texts[:n], batch_size=256); sync(); ts.append(time.perf_counter() - t0)
    return np.mean(ts)
t_all  = time_ce(len(texts)); t_10 = time_ce(10)
t_bi   = np.mean([baseline_retrieve(q)[2] for _ in range(20)])
per_pair = t_all / len(texts)
scale = pd.DataFrame({
  "Strategy": [f"Bi-Encoder + FAISS (Top-10)", "Bi-Encoder Top-10 + Cross-Encoder on 10", f"Cross-Encoder on all {len(texts)} chunks (this doc)",
               "Cross-Encoder on 100,000 chunks (extrapolated)", "Cross-Encoder on 10,000,000 chunks (extrapolated)"],
  "Query latency": [f"{t_bi*1000:.1f} ms", f"{(t_bi+t_10)*1000:.1f} ms", f"{t_all*1000:.0f} ms",
                    f"{per_pair*1e5:.0f} s (~{per_pair*1e5/60:.1f} min)", f"{per_pair*1e7/3600:.1f} h"]})
print(f"per (query, chunk) CE pair ~ {per_pair*1000:.2f} ms")
scale

## 8. Export everything needed for the report
Writes a `results/` folder (also zipped as `results.zip` and downloaded in Colab):

| File | Use in the report |
|---|---|
| `rank_shifts.csv` | Section 4.2 - chunk X moved from rank a -> b for every query |
| `full_rankings.csv` | Top-10 before/after re-ranking with cosine and cross-encoder scores, page and gold flag |
| `top3_chunks.md` | The exact passages the LLM saw in Pipeline 1 vs Pipeline 2 |
| `answers.md` / `answers.csv` | Answer 1 vs Answer 2 side by side with the expected answer and a keyword check |
| `latency.csv`, `latency_summary.csv` | Section 4.3 - latency table |
| `scaling.csv` | Section 4.4 - cost of cross-encoding the whole corpus (measured + extrapolated) |
| `chunks.json`, `ocr_text_rapidocr.json`, `run_config.json` | Corpus, OCR text and settings used (reproducibility) |

In [ ]:
# Heuristic key-fact check: does the answer contain the fact that the gold chunk supports?  (verify manually!)
MUST = {"Q1": r"no[- ]cost|free of cost|without cost|free",  "Q2": r"whatsapp",
        "Q3": r"\bsmes?\b|msmes?",                          "Q4": r"relax",
        "Q5": r"\bno\b.*(g2c|service)|\bnone\b|not mandate|no g2c|no service",
        "Q6": r"user department|each department|every department", "Q7": r"third[- ]party|landowner|developer",
        "Q8": r"unless|mou",                                         "Q9": r"stree nidhi|shg|beneficiar"}
hit = lambda qid, ans: bool(re.search(MUST[qid], ans, re.I | re.S))

OUT = "results"; os.makedirs(OUT, exist_ok=True)

# rank shifts
rank_df.to_csv(f"{OUT}/rank_shifts.csv", index=False)

# full top-10 rankings, before and after
rows = []
for r in results:
    rr_rank = {i: k + 1 for k, i in enumerate(r["rr_ids"])}
    ce_score = dict(zip(r["rr_ids"], r["rr_scores"]))
    for k, (i, s_) in enumerate(zip(r["base_ids"], r["base_scores"]), 1):
        rows.append(dict(query_id=r["id"], category=r["cat"], query=r["q"], chunk_id=i, page=chunks[i]["page"],
                         baseline_rank=k, cosine=round(s_, 4), rerank_rank=rr_rank[i], ce_score=round(ce_score[i], 3),
                         rank_change=k - rr_rank[i], is_gold=i in r["gold_ids"],
                         in_top3_baseline=k <= TOP_N, in_top3_rerank=rr_rank[i] <= TOP_N, text=texts[i]))
pd.DataFrame(rows).to_csv(f"{OUT}/full_rankings.csv", index=False)

# top-3 context given to the LLM, per pipeline
with open(f"{OUT}/top3_chunks.md", "w") as f:
    for r in results:
        f.write(f"## {r['id']} - {r['cat']}\n**Query:** {r['q']}\n\n**Gold chunk(s):** {r['gold_ids']}\n\n")
        for name, ids, sc in (("Pipeline 1 (baseline, cosine)", r["base_ids"], r["base_scores"]),
                              ("Pipeline 2 (re-ranked, cross-encoder)", r["rr_ids"], r["rr_scores"])):
            f.write(f"### {name}\n")
            for k, (i, s_) in enumerate(zip(ids[:TOP_N], sc[:TOP_N]), 1):
                f.write(f"{k}. chunk #{i} (p.{chunks[i]['page']}, score {s_:.3f}){' **[GOLD]**' if i in r['gold_ids'] else ''}: {texts[i]}\n")
            f.write("\n")

# answers
ans_rows = [dict(query_id=r["id"], category=r["cat"], query=r["q"], expected=r["truth"],
                 answer_1_baseline=r["ans1"], answer_2_rerank=r["ans2"],
                 gold_in_top3_baseline=any(v and v <= TOP_N for v in r["base_gold_rank"].values()),
                 gold_in_top3_rerank=any(v and v <= TOP_N for v in r["rr_gold_rank"].values()),
                 keyfact_in_answer_1=hit(r["id"], r["ans1"]), keyfact_in_answer_2=hit(r["id"], r["ans2"])) for r in results]
ans_df = pd.DataFrame(ans_rows); ans_df.to_csv(f"{OUT}/answers.csv", index=False)
with open(f"{OUT}/answers.md", "w") as f:
    for a in ans_rows:
        f.write(f"## {a['query_id']} - {a['category']}\n**Q:** {a['query']}\n\n**Expected:** {a['expected']}\n\n"
                f"**Answer 1 (baseline)** [gold in top-3: {a['gold_in_top3_baseline']}, key fact: {a['keyfact_in_answer_1']}]: {a['answer_1_baseline']}\n\n"
                f"**Answer 2 (re-rank)** [gold in top-3: {a['gold_in_top3_rerank']}, key fact: {a['keyfact_in_answer_2']}]: {a['answer_2_rerank']}\n\n---\n")

# latency + scaling
lat_df.to_csv(f"{OUT}/latency.csv", index=False)
num = lat_df[lat_df["Q"] != "mean"].iloc[:, 1:].astype(float)
pd.DataFrame({"metric": num.columns, "mean": num.mean().round(2).values, "median": num.median().round(2).values,
              "min": num.min().round(2).values, "max": num.max().round(2).values}).to_csv(f"{OUT}/latency_summary.csv", index=False)
scale.to_csv(f"{OUT}/scaling.csv", index=False)

json.dump(chunks, open(f"{OUT}/chunks.json", "w"), indent=1)
shutil.copy(OCR_CACHE, f"{OUT}/ocr_text_rapidocr.json")
json.dump(dict(ocr="RapidOCR PP-OCRv6 medium det+rec, 200 dpi", bi_encoder=BI_NAME, cross_encoder=CE_NAME, llm=LLM_NAME, llm_4bit=USE_4BIT, llm_dtype=str(LLM_DTYPE), encoder_dtype=str(ENC_DTYPE), device=DEVICE, gpu_mem_gb=round(GPU_MEM_GB, 1),
               peak_gpu_mem_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE == "cuda" else None, top_k=TOP_K, top_n=TOP_N,
               n_chunks=len(chunks), chunk_max_chars=450, overlap_sentences=1, latency_reps=REPS, ce_all_chunks_s=t_all,
               ce_10_chunks_s=t_10, ce_ms_per_pair=per_pair * 1000, system_prompt=SYSTEM,
               gpu=GPU_NAME),
          open(f"{OUT}/run_config.json", "w"), indent=1)

shutil.make_archive("results", "zip", OUT)
print(sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download("results.zip")
except ImportError:
    pass
ans_df[["query_id", "gold_in_top3_baseline", "gold_in_top3_rerank", "keyfact_in_answer_1", "keyfact_in_answer_2"]]

## 9. How the outputs map to the report questions
1. **Architecture comparison** - conceptual; no file needed (optionally cite `ce_score` vs `cosine` columns in `full_rankings.csv` to show the different score scales).
2. **Rank shifts + Answer 1 vs 2** - `rank_shifts.csv`, `full_rankings.csv`, `top3_chunks.md`, `answers.md`. Read each pair of answers and decide: hallucination / omitted clause / misstated rule / unchanged. The `keyfact_*` columns are only a keyword heuristic.
3. **Latency** - `latency.csv`, `latency_summary.csv` (retrieval stage: mean of 20 runs; generation: single run).
4. **Scaling** - `scaling.csv` and `ce_ms_per_pair` in `run_config.json`.